In [1]:
import sys
!{sys.executable} -m pip install transformers torch

In [2]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv("../data/processed/tweets_cleaned.csv")

# Türkçe sentiment modeli
sentiment_model = pipeline(
    "text-classification",
    model="savasy/bert-base-turkish-sentiment-cased",
    truncation=True,
    max_length=512
)

print(f"✅ {len(df)} satır yüklendi, model hazır")

C:\Users\Eser\anaconda3\envs\spark\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6656.15it/s]


✅ 1695 satır yüklendi, model hazır


In [3]:
def analyze(text):
    try:
        result = sentiment_model(str(text)[:512])[0]
        label = result["label"]   # "positive" veya "negative"
        score = result["score"]   # güven skoru (0-1)
        return label, score
    except:
        return "notr", 0.5

print("Skorlanıyor... (~5-10 dk sürebilir)")

results = df["clean_text"].apply(analyze)
df["label_raw"] = results.apply(lambda x: x[0])
df["confidence"] = results.apply(lambda x: x[1])

print("✅ Bitti")

Skorlanıyor... (~5-10 dk sürebilir)
✅ Bitti


In [4]:
def map_label(row):
    if row["confidence"] < 0.65:   # düşük güven → nötr
        return "Nötr"
    elif row["label_raw"] == "positive":
        return "Pozitif"
    else:
        return "Negatif"

df["sentiment"] = df.apply(map_label, axis=1)

print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True).mul(100).round(1))
df[["text", "sentiment", "confidence"]].head(10)

sentiment
Negatif    1019
Pozitif     498
Nötr        178
Name: count, dtype: int64
sentiment
Negatif    60.1
Pozitif    29.4
Nötr       10.5
Name: proportion, dtype: float64


,text,sentiment,confidence
0,Yav sktrin gidin amk sokacam artık da,Nötr,0.538618
1,bu lesley magassa masalı da 1-2 hafta sorer ge...,Negatif,0.853064
2,#DursunÖzbekParalarNerede,Negatif,0.810934
3,Can -Lesley -Reijnders Can 10 no Lesley 6 Reij...,Pozitif,0.985296
4,Samiyen haberden gör burda gir iyi la 😀😀,Pozitif,0.844890
5,aniden düşen güzel haberler,Nötr,0.575131
6,La siz yalan soyluyonuz inanmiyom hicbirinize,Negatif,0.726682
7,Şu an yaşadıklarımızın rüya olduğunu eylül baş...,Nötr,0.581698
8,Yav şu 23 yaş üstü kontenjanı iyi birini alıp ...,Negatif,0.996077
9,Yalan olduğunu bilsem de inanmak istiyorum,Pozitif,0.953831


In [5]:
df.to_csv("../data/processed/tweets_sentiment.csv", index=False, encoding="utf-8-sig")
print(f"✅ Kaydedildi ({len(df)} satır)")

✅ Kaydedildi (1695 satır)
